# WC_MERCURY_BADGEEVENT_F ETL - ODI to Databricks Migration

**Original Package:** WC_MERCURY_BADGEEVENT_F Load

**Source Schema:** `workspace.prxbi_ts_sep` (Staging/Transient)

**Target Schema:** `workspace.prxbi_dw_sep` (Data Warehouse)

**Source Table:** `WC_MERCURY_BADGEEVENT_TS`

**Target Table:** `WC_MERCURY_BADGEEVENT_F`

**Detection Strategy:** NONE (all qualifying rows processed every run)

**DATASOURCE_NUM_ID:** 380

**Load Pattern:** Incremental Extract with Full Insert/Update (no change detection on target)

---

## Step 1: Define Variables as Temp Views

Extract ETL control parameters from `WC_ETL_PARAMETERS`.

In [ ]:
%sql
-- Variable: V_ETL_JOB_TYPE
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_JOB_TYPE AS
SELECT 'EOD' AS V_ETL_JOB_TYPE

In [ ]:
%sql
-- Variable: V_ETL_LAST_EXTRACT_TIME
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_LAST_EXTRACT_TIME AS
SELECT etl_last_extract_time AS V_ETL_LAST_EXTRACT_TIME
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT V_ETL_JOB_TYPE FROM VAR_ETL_JOB_TYPE)

In [ ]:
%sql
-- Variable: V_ETL_CURRENT_EXTRACT_TIME
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_CURRENT_EXTRACT_TIME AS
SELECT etl_current_extract_time AS V_ETL_CURRENT_EXTRACT_TIME
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT V_ETL_JOB_TYPE FROM VAR_ETL_JOB_TYPE)

In [ ]:
%sql
-- Variable: V_ETL_PROC_WID
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_PROC_WID AS
SELECT ROW_WID AS V_ETL_PROC_WID
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT V_ETL_JOB_TYPE FROM VAR_ETL_JOB_TYPE)

In [ ]:
%sql
-- Display ETL parameters for verification
SELECT
  'V_ETL_LAST_EXTRACT_TIME' AS parameter,
  CAST(V_ETL_LAST_EXTRACT_TIME AS STRING) AS value
FROM VAR_ETL_LAST_EXTRACT_TIME
UNION ALL
SELECT
  'V_ETL_CURRENT_EXTRACT_TIME' AS parameter,
  CAST(V_ETL_CURRENT_EXTRACT_TIME AS STRING) AS value
FROM VAR_ETL_CURRENT_EXTRACT_TIME
UNION ALL
SELECT
  'V_ETL_PROC_WID' AS parameter,
  CAST(V_ETL_PROC_WID AS STRING) AS value
FROM VAR_ETL_PROC_WID

---

## Step 2: C$ Work Table as Temp View (Source Extract with Deduplication)

Extract from `WC_MERCURY_BADGEEVENT_TS` with complex deduplication logic:
1. Inner subquery groups by natural key columns and finds MAX(INT_INSERT_DATE) within the extract window.
2. Joins back to source to retrieve EVENTDATE and CREATEDDATE for those grouped records.
3. Applies ROW_NUMBER() partitioned by natural key, ordered by EVENTDATE DESC, to pick the latest event per key.
4. Filters to RN=1 to keep only the most recent event per natural key combination.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW C$_BADGEEVENT AS
SELECT
  INLINE_VIEW.EVENTEDITIONGBSCODE,
  INLINE_VIEW.EVENTTYPE,
  INLINE_VIEW.BADGEID,
  INLINE_VIEW.SOURCE,
  INLINE_VIEW.PRODUCTCODE,
  INLINE_VIEW.CUSTOMERTYPE,
  INLINE_VIEW.EVENTDATE,
  INLINE_VIEW.CREATEDDATE
FROM (
  SELECT
    AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
    AMERCURY_BADGEEVENT_TS.EVENTTYPE,
    AMERCURY_BADGEEVENT_TS.BADGEID,
    AMERCURY_BADGEEVENT_TS.SOURCE,
    AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
    AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE,
    AMERCURY_BADGEEVENT_TS.EVENTDATE,
    AMERCURY_BADGEEVENT_TS.CREATEDDATE,
    ROW_NUMBER() OVER (
      PARTITION BY
        AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
        AMERCURY_BADGEEVENT_TS.EVENTTYPE,
        AMERCURY_BADGEEVENT_TS.BADGEID,
        AMERCURY_BADGEEVENT_TS.SOURCE,
        AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
        AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE
      ORDER BY AMERCURY_BADGEEVENT_TS.EVENTDATE DESC
    ) AS RN
  FROM (
    SELECT
      AMERCURY_BADGEEVENT_TS1.EVENTEDITIONGBSCODE,
      AMERCURY_BADGEEVENT_TS1.EVENTTYPE,
      AMERCURY_BADGEEVENT_TS1.BADGEID,
      AMERCURY_BADGEEVENT_TS1.SOURCE,
      AMERCURY_BADGEEVENT_TS1.PRODUCTCODE,
      AMERCURY_BADGEEVENT_TS1.CUSTOMERTYPE,
      MAX(AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE) AS INT_INSERT_DATE
    FROM workspace.prxbi_ts_sep.WC_MERCURY_BADGEEVENT_TS AMERCURY_BADGEEVENT_TS1
    WHERE AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE > (SELECT V_ETL_LAST_EXTRACT_TIME FROM VAR_ETL_LAST_EXTRACT_TIME)
      AND AMERCURY_BADGEEVENT_TS1.INT_INSERT_DATE <= (SELECT V_ETL_CURRENT_EXTRACT_TIME FROM VAR_ETL_CURRENT_EXTRACT_TIME)
    GROUP BY
      AMERCURY_BADGEEVENT_TS1.EVENTEDITIONGBSCODE,
      AMERCURY_BADGEEVENT_TS1.EVENTTYPE,
      AMERCURY_BADGEEVENT_TS1.BADGEID,
      AMERCURY_BADGEEVENT_TS1.SOURCE,
      AMERCURY_BADGEEVENT_TS1.PRODUCTCODE,
      AMERCURY_BADGEEVENT_TS1.CUSTOMERTYPE
  ) AMERCURY_BADGEEVENT_TS1_1
  INNER JOIN workspace.prxbi_ts_sep.WC_MERCURY_BADGEEVENT_TS AMERCURY_BADGEEVENT_TS
    ON AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE = AMERCURY_BADGEEVENT_TS1_1.EVENTEDITIONGBSCODE
    AND AMERCURY_BADGEEVENT_TS1_1.EVENTTYPE = AMERCURY_BADGEEVENT_TS.EVENTTYPE
    AND AMERCURY_BADGEEVENT_TS1_1.BADGEID = AMERCURY_BADGEEVENT_TS.BADGEID
    AND AMERCURY_BADGEEVENT_TS1_1.SOURCE = AMERCURY_BADGEEVENT_TS.SOURCE
    AND AMERCURY_BADGEEVENT_TS1_1.PRODUCTCODE = AMERCURY_BADGEEVENT_TS.PRODUCTCODE
    AND AMERCURY_BADGEEVENT_TS1_1.CUSTOMERTYPE = AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE
    AND AMERCURY_BADGEEVENT_TS1_1.INT_INSERT_DATE = AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE
  WHERE AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE > (SELECT V_ETL_LAST_EXTRACT_TIME FROM VAR_ETL_LAST_EXTRACT_TIME)
    AND AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE <= (SELECT V_ETL_CURRENT_EXTRACT_TIME FROM VAR_ETL_CURRENT_EXTRACT_TIME)
  GROUP BY
    AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,
    AMERCURY_BADGEEVENT_TS.EVENTTYPE,
    AMERCURY_BADGEEVENT_TS.BADGEID,
    AMERCURY_BADGEEVENT_TS.SOURCE,
    AMERCURY_BADGEEVENT_TS.PRODUCTCODE,
    AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE,
    AMERCURY_BADGEEVENT_TS.EVENTDATE,
    AMERCURY_BADGEEVENT_TS.CREATEDDATE
) INLINE_VIEW
WHERE INLINE_VIEW.RN = 1

In [ ]:
%sql
-- Verify C$ record count
SELECT COUNT(*) AS c_dollar_record_count FROM C$_BADGEEVENT

---

## Step 3: I$ Flow Table as Temp View (Transformations + Lookups)

Detection Strategy = NONE: All source rows are inserted with IND_UPDATE = 'I'.

Lookups performed:
- **WC_EVENT_ED_D**: Matches EVENTEDITIONGBSCODE to `RPAD(EVENT_ALPHA_CODE,5,'-') || EVENT_EDITION_CODE` to get EVENT_EDITION_WID and OBU_WID.
- **WC_BADGE_DETAILS_D**: Matches BADGEID to BADGE_ID to get BADGE_WID (ROW_WID).
- **WC_BADGE_PRODUCT_D**: Matches PRODUCTCODE to SKU to get PRODUCT_WID (MAX ROW_WID).
- **WC_EVENT_D**: Matches EVENT_INTEGRATION_ID from WC_EVENT_ED_D to get EVENT_WID.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW I$_BADGEEVENT AS
SELECT
  FILTER2_A_1.EVENTEDITIONGBSCODE || '~' || FILTER2_A_1.EVENTTYPE || '~' || FILTER2_A_1.BADGEID || '~' || FILTER2_A_1.SOURCE || '~' || FILTER2_A_1.PRODUCTCODE || '~' || FILTER2_A_1.CUSTOMERTYPE AS INTEGRATION_ID,
  FILTER2_A_1.EVENTEDITIONGBSCODE,
  FILTER2_A_1.EVENTTYPE,
  FILTER2_A_1.BADGEID,
  FILTER2_A_1.SOURCE,
  FILTER2_A_1.PRODUCTCODE,
  FILTER2_A_1.CUSTOMERTYPE,
  FILTER2_A_1.EVENTDATE,
  FILTER2_A_1.CREATEDDATE,
  COALESCE(WC_BADGE_DETAILS_D.ROW_WID, 0) AS BADGE_WID,
  COALESCE(WC_EVENT_ED_D_1.ROW_WID, 0) AS EVENT_EDITION_WID,
  COALESCE(WC_EVENT_ED_D_1.OBU_WID, 0) AS OBU_WID,
  COALESCE(WC_BADGE_PRODUCT_D_SQ_BADGEP_1.ROW_WID_1, 0) AS PRODUCT_WID,
  COALESCE(WC_EVENT_D.ROW_WID, 0) AS EVENT_WID,
  'I' AS IND_UPDATE
FROM (
  SELECT * FROM C$_BADGEEVENT
) FILTER2_A_1
LEFT OUTER JOIN (
  -- Lookup: WC_EVENT_ED_D for EVENT_EDITION_WID and OBU_WID
  SELECT ROW_WID, EVENT_EDITION_CODE, EVENT_ALPHA_CODE, OBU_WID, EVENT_INTEGRATION_ID
  FROM workspace.prxbi_dw_sep.WC_EVENT_ED_D
) WC_EVENT_ED_D_1
  ON FILTER2_A_1.EVENTEDITIONGBSCODE = (RPAD(WC_EVENT_ED_D_1.EVENT_ALPHA_CODE, 5, '-') || WC_EVENT_ED_D_1.EVENT_EDITION_CODE)
LEFT OUTER JOIN workspace.prxbi_dw_sep.WC_BADGE_DETAILS_D
  ON FILTER2_A_1.BADGEID = WC_BADGE_DETAILS_D.BADGE_ID
LEFT OUTER JOIN (
  -- Lookup: WC_BADGE_PRODUCT_D for PRODUCT_WID
  SELECT MAX(ROW_WID) AS ROW_WID, SKU, MAX(ROW_WID) AS ROW_WID_1, SKU AS SKU_1
  FROM workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D
  GROUP BY SKU
) WC_BADGE_PRODUCT_D_SQ_BADGEP_1
  ON WC_BADGE_PRODUCT_D_SQ_BADGEP_1.SKU_1 = FILTER2_A_1.PRODUCTCODE
LEFT OUTER JOIN workspace.prxbi_dw_sep.WC_EVENT_D
  ON WC_EVENT_ED_D_1.EVENT_INTEGRATION_ID = WC_EVENT_D.INTEGRATION_ID
WHERE (1=1)

In [ ]:
%sql
-- Verify I$ record count and breakdown
SELECT
  COUNT(*) AS i_dollar_record_count,
  SUM(CASE WHEN BADGE_WID = 0 THEN 1 ELSE 0 END) AS unmatched_badge,
  SUM(CASE WHEN EVENT_EDITION_WID = 0 THEN 1 ELSE 0 END) AS unmatched_event_edition,
  SUM(CASE WHEN PRODUCT_WID = 0 THEN 1 ELSE 0 END) AS unmatched_product,
  SUM(CASE WHEN EVENT_WID = 0 THEN 1 ELSE 0 END) AS unmatched_event
FROM I$_BADGEEVENT

---

## Step 4: PK Duplicate Check and Error Handling

Check for duplicate INTEGRATION_IDs in I$. If duplicates exist, log them and remove them
by keeping only the first occurrence (using ROW_NUMBER).

In [ ]:
%sql
-- Check for duplicate INTEGRATION_IDs
CREATE OR REPLACE TEMPORARY VIEW E$_BADGEEVENT_DUPLICATES AS
SELECT
  INTEGRATION_ID,
  COUNT(*) AS DUP_COUNT
FROM I$_BADGEEVENT
GROUP BY INTEGRATION_ID
HAVING COUNT(*) > 1

In [ ]:
%sql
-- Display duplicate count
SELECT COUNT(*) AS duplicate_integration_id_count FROM E$_BADGEEVENT_DUPLICATES

In [ ]:
%sql
-- Deduplicate I$ by keeping only one row per INTEGRATION_ID
-- Use ROW_NUMBER to pick the first row per INTEGRATION_ID (ordered by EVENTDATE DESC)
CREATE OR REPLACE TEMPORARY VIEW I$_BADGEEVENT_DEDUPED AS
SELECT
  INTEGRATION_ID,
  EVENTEDITIONGBSCODE,
  EVENTTYPE,
  BADGEID,
  SOURCE,
  PRODUCTCODE,
  CUSTOMERTYPE,
  EVENTDATE,
  CREATEDDATE,
  BADGE_WID,
  EVENT_EDITION_WID,
  OBU_WID,
  PRODUCT_WID,
  EVENT_WID,
  IND_UPDATE
FROM (
  SELECT
    I$_BADGEEVENT.*,
    ROW_NUMBER() OVER (PARTITION BY INTEGRATION_ID ORDER BY EVENTDATE DESC) AS DEDUP_RN
  FROM I$_BADGEEVENT
) deduped
WHERE DEDUP_RN = 1

In [ ]:
%sql
-- Verify deduped I$ record count
SELECT COUNT(*) AS i_dollar_deduped_count FROM I$_BADGEEVENT_DEDUPED

---

## Step 5: MERGE for Updates (Existing Rows)

Update existing rows in `WC_MERCURY_BADGEEVENT_F` where the INTEGRATION_ID already exists.
Since DETECTION_STRATEGY = NONE, all matching rows are updated regardless of whether data changed.
The ODI original uses `UPDATE ... SET ... WHERE EXISTS`, converted here to `MERGE INTO ... WHEN MATCHED`.

In [ ]:
%sql
MERGE INTO workspace.prxbi_dw_sep.WC_MERCURY_BADGEEVENT_F AS T
USING (
  SELECT
    S.INTEGRATION_ID,
    S.EVENTEDITIONGBSCODE,
    S.EVENTTYPE,
    S.BADGEID,
    S.SOURCE,
    S.PRODUCTCODE,
    S.CUSTOMERTYPE,
    S.EVENTDATE,
    S.CREATEDDATE,
    S.BADGE_WID,
    S.EVENT_EDITION_WID,
    S.OBU_WID,
    S.PRODUCT_WID,
    S.EVENT_WID,
    P.V_ETL_PROC_WID AS ETL_PROC_WID
  FROM I$_BADGEEVENT_DEDUPED S
  CROSS JOIN VAR_ETL_PROC_WID P
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
WHEN MATCHED THEN UPDATE SET
  T.EVENTEDITIONGBSCODE = S.EVENTEDITIONGBSCODE,
  T.EVENTTYPE = S.EVENTTYPE,
  T.BADGEID = S.BADGEID,
  T.SOURCE = S.SOURCE,
  T.PRODUCTCODE = S.PRODUCTCODE,
  T.CUSTOMERTYPE = S.CUSTOMERTYPE,
  T.EVENTDATE = S.EVENTDATE,
  T.CREATEDDATE = S.CREATEDDATE,
  T.BADGE_WID = S.BADGE_WID,
  T.EVENT_EDITION_WID = S.EVENT_EDITION_WID,
  T.OBU_WID = S.OBU_WID,
  T.PRODUCT_WID = S.PRODUCT_WID,
  T.EVENT_WID = S.EVENT_WID,
  T.ETL_PROC_WID = S.ETL_PROC_WID,
  T.W_UPDATE_DT = current_timestamp()

---

## Step 6: INSERT for New Rows

Insert rows from I$ where the INTEGRATION_ID does not yet exist in the target table.
ROW_WID is generated using `row_number() OVER (ORDER BY INTEGRATION_ID) + COALESCE(MAX(ROW_WID), 0)` from the target.

In [ ]:
%sql
INSERT INTO workspace.prxbi_dw_sep.WC_MERCURY_BADGEEVENT_F (
  ROW_WID,
  INTEGRATION_ID,
  EVENTEDITIONGBSCODE,
  EVENTTYPE,
  BADGEID,
  SOURCE,
  PRODUCTCODE,
  CUSTOMERTYPE,
  EVENTDATE,
  CREATEDDATE,
  BADGE_WID,
  EVENT_EDITION_WID,
  OBU_WID,
  PRODUCT_WID,
  EVENT_WID,
  ETL_PROC_WID,
  W_INSERT_DT,
  W_UPDATE_DT
)
SELECT
  ROW_NUMBER() OVER (ORDER BY S.INTEGRATION_ID) + COALESCE((SELECT MAX(ROW_WID) FROM workspace.prxbi_dw_sep.WC_MERCURY_BADGEEVENT_F), 0) AS ROW_WID,
  S.INTEGRATION_ID,
  S.EVENTEDITIONGBSCODE,
  S.EVENTTYPE,
  S.BADGEID,
  S.SOURCE,
  S.PRODUCTCODE,
  S.CUSTOMERTYPE,
  S.EVENTDATE,
  S.CREATEDDATE,
  S.BADGE_WID,
  S.EVENT_EDITION_WID,
  S.OBU_WID,
  S.PRODUCT_WID,
  S.EVENT_WID,
  P.V_ETL_PROC_WID AS ETL_PROC_WID,
  current_timestamp() AS W_INSERT_DT,
  current_timestamp() AS W_UPDATE_DT
FROM I$_BADGEEVENT_DEDUPED S
CROSS JOIN VAR_ETL_PROC_WID P
WHERE NOT EXISTS (
  SELECT 1
  FROM workspace.prxbi_dw_sep.WC_MERCURY_BADGEEVENT_F T2
  WHERE T2.INTEGRATION_ID = S.INTEGRATION_ID
)

---

## Step 7: ETL Tracking Update

Update the ETL parameters table to reflect successful completion.

In [ ]:
%sql
-- Update ETL parameters: set last extract time = current extract time
UPDATE workspace.prxbi_dw_sep.wc_etl_parameters
SET etl_last_extract_time = (
  SELECT V_ETL_CURRENT_EXTRACT_TIME FROM VAR_ETL_CURRENT_EXTRACT_TIME
)
WHERE ETL_JOB_TYPE = (SELECT V_ETL_JOB_TYPE FROM VAR_ETL_JOB_TYPE)

In [ ]:
%sql
-- Validate target table summary
SELECT
  'WC_MERCURY_BADGEEVENT_F' AS table_name,
  COUNT(*) AS total_records,
  COUNT(DISTINCT INTEGRATION_ID) AS unique_integration_ids,
  MAX(W_UPDATE_DT) AS last_update_time
FROM workspace.prxbi_dw_sep.WC_MERCURY_BADGEEVENT_F

---

## Step 8: Cleanup

Drop all temporary views created during this ETL run.

In [ ]:
%sql
DROP VIEW IF EXISTS C$_BADGEEVENT;
DROP VIEW IF EXISTS I$_BADGEEVENT;
DROP VIEW IF EXISTS I$_BADGEEVENT_DEDUPED;
DROP VIEW IF EXISTS E$_BADGEEVENT_DUPLICATES;
DROP VIEW IF EXISTS VAR_ETL_LAST_EXTRACT_TIME;
DROP VIEW IF EXISTS VAR_ETL_CURRENT_EXTRACT_TIME;
DROP VIEW IF EXISTS VAR_ETL_PROC_WID;
DROP VIEW IF EXISTS VAR_ETL_JOB_TYPE